# FANet Phase-7B: Detached Soft-OR Gate — Smoke Test & Multi-Seed

**Mục tiêu:** Kiểm chứng liệu `MixPool(soft_or + detach_feedback=True)` có phá vỡ Feedback Trap và phục hồi tác dụng của Tversky Loss trong việc giảm FP không.

| Cell | Feedback | Loss | Gating | Detach | Kỳ vọng |
|---|---|---|---|---|---|
| M11 (baseline) | ON | Tversky | binary | False | FP cao (Trap active) |
| **M12 (Phase-7B)** | ON | Tversky | soft_or | **True** | FP giảm (Trap severed) |

---
## Cấu trúc Notebook
- **Part 1** — Smoke Test M12 (1 seed, 200ep) — Fail-Fast validation
- **Part 2** — Inline Diagnostics (BN Drift, CosSim, Saturation)
- **Part 3** — Multi-seed loop M11 vs M12 (comment ra để bật sau Smoke Test OK)


In [ ]:
# ════════════════════════════════════════════════════════════════
# 0. SETUP — Install, mount data, set paths
# ════════════════════════════════════════════════════════════════
import os, sys, subprocess

# ── Kaggle dataset paths ──────────────────────────────────────────
DATASET_PATH = "/kaggle/input/kvasir-sessile-polyp-dataset/sessile-main-Kvasir-SEG"
REPO_PATH    = "/kaggle/input/fanet-repo"          # mounted via Kaggle Dataset
WORK_DIR     = "/kaggle/working"

# Add src to Python path
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
sys.path.insert(0, REPO_PATH)

# ── Output directories ────────────────────────────────────────────
CKPT_DIR    = f"{WORK_DIR}/checkpoints_phase7b"
LOG_DIR     = f"{WORK_DIR}/logs_phase7b"
RESULT_DIR  = f"{WORK_DIR}/results_phase7b"
DIAG_DIR    = f"{WORK_DIR}/diagnostics_phase7b"
for d in [CKPT_DIR, LOG_DIR, RESULT_DIR, DIAG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Setup complete.")
print(f"  Dataset : {DATASET_PATH}")
print(f"  Repo    : {REPO_PATH}")
print(f"  Working : {WORK_DIR}")
!nvidia-smi | head -20

In [ ]:
import torch, numpy as np
from torch.utils.data import DataLoader
import albumentations as A

from fanet.config import load_config
from fanet.data import DATASET, load_data, rle_batch_to_tensor
from fanet.losses import Phase6AsymmetricBCELoss
from fanet.models import FANet
from fanet.utils import seeding, shuffling, init_mask, epoch_time, rle_encode, print_and_save

# Inline copies of train/evaluate to avoid CLI dependency
# (same logic as scripts/train.py)
from scripts_inline import train_epoch, evaluate_epoch   # defined in cell below

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# ── Inline train/evaluate (mirrors scripts/train.py) ─────────────
import time

def train_epoch(model, loader, mask_rle, optimizer, loss_fn, device, size, no_feedback=False):
    from fanet.data import rle_batch_to_tensor
    epoch_loss = 0; return_mask = []
    model.train()
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device, dtype=torch.float32), y.to(device, dtype=torch.float32)
        b = y.shape[0]
        m = (torch.zeros(b, 1, *size) if no_feedback
             else rle_batch_to_tensor(mask_rle, i*b, b, size))
        m = m.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model([x, m]), y)
        loss.backward(); optimizer.step()
        with torch.no_grad():
            pred = torch.sigmoid(model([x, m])).cpu().numpy()
            for py in pred:
                py = (np.squeeze(py) > 0.5).astype(np.uint8)
                from fanet.utils import rle_encode
                return_mask.append(rle_encode(py))
        epoch_loss += loss.item()
    return epoch_loss / len(loader), return_mask

def evaluate_epoch(model, loader, mask_rle, loss_fn, device, size, no_feedback=False):
    from fanet.data import rle_batch_to_tensor
    epoch_loss = 0; return_mask = []
    bin_dice = bin_prec = bin_rec = bin_fpr = 0.0
    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device, dtype=torch.float32), y.to(device, dtype=torch.float32)
            b = y.shape[0]
            m = (torch.zeros(b, 1, *size) if no_feedback
                 else rle_batch_to_tensor(mask_rle, i*b, b, size))
            m = m.to(device)
            pred_logit = model([x, m])
            epoch_loss += loss_fn(pred_logit, y).item()
            pred = torch.sigmoid(pred_logit).cpu().numpy()
            for py, gy in zip(pred, y.cpu().numpy()):
                py = (np.squeeze(py) > 0.5).astype(np.uint8)
                gb = (np.squeeze(gy) > 0.5).astype(np.uint8)
                from fanet.utils import rle_encode
                return_mask.append(rle_encode(py))
                tp = (py & gb).sum(); pp = py.sum(); gp = gb.sum()
                fp = (py & (1-gb)).sum()
                bin_dice += 2*tp/(pp+gp+1e-15)
                bin_prec += tp/(pp+1e-15)
                bin_rec  += tp/(gp+1e-15)
                bin_fpr  += fp/(gb.size+1e-15)
    n = len(loader.dataset)
    return (epoch_loss/len(loader), return_mask,
            {"bin_dice": bin_dice/n, "bin_prec": bin_prec/n,
             "bin_rec": bin_rec/n,  "bin_fpr": bin_fpr/n})

print("Inline train/evaluate defined.")

---
## Part 1 — Smoke Test: M12 (soft_or + detach + Tversky), Seed 42

**Fail-Fast criteria (check at epoch 40):**
- `valid_loss` đang hội tụ (không plateau từ epoch 1)
- `bin_dice` > 0.05 (không collapse như TB trong Phase 5)
- Không có NaN trong loss

Nếu epoch 40 OK → tiếp tục 200 epochs. Nếu fail → dừng, debug.

In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 1: SMOKE TEST — M12 (soft_or + detach + Tversky)
# ════════════════════════════════════════════════════════════════
SMOKE_SEED   = 42
SMOKE_EPOCHS = 200       # full run; set to 40 for quick sanity check
SIZE         = (256, 256)
BATCH_SIZE   = 2
LR           = 1e-4
TVERSKY_ALPHA = 0.7
TVERSKY_BETA  = 0.3
FAIL_FAST_EP  = 40       # epoch to check Fail-Fast criteria

M12_CKPT = f"{CKPT_DIR}/M12_soft_or_detach_tversky_seed{SMOKE_SEED}.pth"
M12_LOG  = f"{LOG_DIR}/M12_seed{SMOKE_SEED}.log"

seeding(SMOKE_SEED)

# ── Data ─────────────────────────────────────────────────────────
(train_x, train_y), (valid_x, valid_y) = load_data(DATASET_PATH)
train_x, train_y = shuffling(train_x, train_y)

aug = A.Compose([
    A.Rotate(limit=35, p=0.3), A.HorizontalFlip(p=0.3),
    A.VerticalFlip(p=0.3),
    A.CoarseDropout(p=0.3, num_holes=10, hole_height=32, hole_width=32),
])
train_loader = DataLoader(DATASET(train_x, train_y, SIZE, transform=aug),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
valid_loader = DataLoader(DATASET(valid_x, valid_y, SIZE, transform=None),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── Model: M12 — THE FIX ─────────────────────────────────────────
model_m12 = FANet(
    gating_mode="soft_or",
    detach_feedback=True,    # KEY: severs Feedback Trap
    dual_path=False,
).to(DEVICE)

print(f"M12 config: gating_mode=soft_or | detach_feedback=True | loss=Tversky(a={TVERSKY_ALPHA})")
print(f"Parameters: {sum(p.numel() for p in model_m12.parameters()):,}")

# ── Loss + Optimizer ─────────────────────────────────────────────
loss_fn   = Phase6AsymmetricBCELoss(alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA, lam=0.5)
optimizer = torch.optim.Adam(model_m12.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

train_mask = init_mask(train_x, SIZE)
valid_mask = init_mask(valid_x, SIZE)
best_loss  = float('inf')

log_header = (f"M12 Smoke Test | seed={SMOKE_SEED} | gating=soft_or | "
              f"detach=True | loss=Tversky(a={TVERSKY_ALPHA})\n")
print_and_save(M12_LOG, log_header)

# ── Training Loop ─────────────────────────────────────────────────
for epoch in range(SMOKE_EPOCHS):
    t0 = time.time()
    tr_loss, tr_mask = train_epoch(
        model_m12, train_loader, train_mask, optimizer, loss_fn, DEVICE, SIZE)
    vl_loss, vl_mask, bm = evaluate_epoch(
        model_m12, valid_loader, valid_mask, loss_fn, DEVICE, SIZE)
    scheduler.step(vl_loss)

    if vl_loss < best_loss:
        best_loss = vl_loss
        torch.save(model_m12.state_dict(), M12_CKPT)
        train_mask = tr_mask
        valid_mask = vl_mask

    mins, secs = epoch_time(t0, time.time())
    row = (f"Ep {epoch+1:03}/{SMOKE_EPOCHS} | {mins}m{secs}s | "
           f"TrLoss={tr_loss:.4f} VlLoss={vl_loss:.4f} | "
           f"Dice={bm['bin_dice']:.4f} FPR={bm['bin_fpr']:.4f}")
    print_and_save(M12_LOG, row)

    # ── Fail-Fast check at epoch 40 ───────────────────────────────
    if (epoch + 1) == FAIL_FAST_EP:
        print(f"\n{'='*55}")
        print(f"FAIL-FAST CHECK @ Epoch {FAIL_FAST_EP}")
        print(f"  bin_dice : {bm['bin_dice']:.4f}  (must be > 0.05)")
        print(f"  bin_fpr  : {bm['bin_fpr']:.4f}")
        print(f"  vl_loss  : {vl_loss:.4f}")
        if bm['bin_dice'] < 0.05:
            print("  [FAIL] Model collapsed! Stop and debug.")
            raise RuntimeError("FAIL-FAST: bin_dice < 0.05 at epoch 40 — model collapsed.")
        else:
            print("  [PASS] Model healthy. Continuing to full 200 epochs.")
        print(f"{'='*55}\n")

print(f"\nSmoke Test done. Best valid loss: {best_loss:.4f}")
print(f"Checkpoint saved: {M12_CKPT}")

---
## Part 2 — Inline Diagnostics

Chạy `diagnose_feedback_drift.py` ngay trong notebook để so sánh M11 (Trap) vs M12 (Fixed). Nếu diagnostics thành công:
- **Saturation M12 < M11** → prediction không còn bị ép cứng
- **CosSim M12 ≈ M00** → feature không bị collapse
- **Grad Norm M12 > M11** → gradient nhánh fmask được phục hồi

In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 2: INLINE DIAGNOSTICS — Compare M11 (Trap) vs M12 (Fixed)
# ════════════════════════════════════════════════════════════════
import cv2, json
import torch.nn.functional as F

# Paths to existing checkpoints (from Phase 6)
M11_CKPT_PATH = "/kaggle/input/fanet-phase6-checkpoints/ckpt_TC.pth"  # FB + Tversky (TRAP)
M00_CKPT_PATH = "/kaggle/input/fanet-phase5-checkpoints/ckpt_T0N.pth" # No-FB + Base
M01_CKPT_PATH = "/kaggle/input/fanet-phase6-checkpoints/ckpt_TD.pth"  # No-FB + Tversky
# M12 checkpoint is from Part 1 above

MODELS_INFO = {
    "M00": {"path": M00_CKPT_PATH, "kwargs": {"gate": "binary"},                        "label": "No-FB + Base"},
    "M01": {"path": M01_CKPT_PATH, "kwargs": {"gate": "binary"},                        "label": "No-FB + Tversky"},
    "M11": {"path": M11_CKPT_PATH, "kwargs": {"gate": "binary"},                        "label": "FB + Tversky [TRAP]"},
    "M12": {"path": M12_CKPT,      "kwargs": {"gating_mode": "soft_or",
                                               "detach_feedback": True},                 "label": "FB + Tversky [FIXED]"},
}

def get_bn_stats(model):
    import torch.nn as nn
    return {n: {"mean": m.running_mean.clone().cpu().numpy(),
                "var":  m.running_var.clone().cpu().numpy()}
            for n, m in model.named_modules() if isinstance(m, nn.BatchNorm2d)}

def kl_div(m1, v1, m0, v0, eps=1e-5):
    v0, v1 = np.maximum(v0, eps), np.maximum(v1, eps)
    return float(np.mean(np.log(v0**.5/v1**.5) + (v1+(m1-m0)**2)/(2*v0) - 0.5))

# ── Load models ──────────────────────────────────────────────────
loaded = {}
bn_stats = {}
for key, info in MODELS_INFO.items():
    if not os.path.exists(info["path"]):
        print(f"[SKIP] {key}: {info['path']} not found")
        continue
    net = FANet(**info["kwargs"]).to(DEVICE)
    net.load_state_dict(torch.load(info["path"], map_location=DEVICE))
    net.eval()
    loaded[key]   = net
    bn_stats[key] = get_bn_stats(net)
    print(f"[OK] Loaded {key}: {info['label']}")

# ── BatchNorm Drift: M11 vs M00, M12 vs M00 ──────────────────────
print("\n--- BN Drift ---")
comparisons = [("M11", "M00", "Trap active"), ("M12", "M00", "Trap fixed")]
bn_summary = {}
for fb_k, ref_k, tag in comparisons:
    if fb_k not in bn_stats or ref_k not in bn_stats:
        continue
    layer_kls = []
    for layer in bn_stats[fb_k]:
        if layer not in bn_stats[ref_k]: continue
        kl = kl_div(bn_stats[fb_k][layer]["mean"], bn_stats[fb_k][layer]["var"],
                    bn_stats[ref_k][layer]["mean"], bn_stats[ref_k][layer]["var"])
        layer_kls.append((layer, kl))
    avg_kl = float(np.mean([k[1] for k in layer_kls]))
    top3   = sorted(layer_kls, key=lambda x: x[1], reverse=True)[:3]
    bn_summary[f"{fb_k}_vs_{ref_k}"] = {"avg_kl": round(avg_kl, 4), "top3": [(k, round(v,2)) for k,v in top3]}
    print(f"  {fb_k} vs {ref_k} [{tag}]: avg KL = {avg_kl:.4f} | top: {top3[0]}")

# ── Feature Collapse & Saturation per model ───────────────────────
print("\n--- Feature Collapse & Saturation ---")
diag = {k: {"cos_sim": [], "saturation": [], "entropy": [], "grad_norm": []} for k in loaded}

# Hooks on e4
hooks = {}; feat_store = {}
for k, net in loaded.items():
    def make_hook(key):
        def fn(m, i, o):
            feat_store[key] = (o[1] if isinstance(o, (tuple, list)) else o).detach()
        return fn
    hooks[k] = net.e4.register_forward_hook(make_hook(k))

val_loader_diag = DataLoader(
    DATASET(valid_x, valid_y, SIZE, transform=None),
    batch_size=1, shuffle=False)

for x_b, y_b in val_loader_diag:
    x_b = x_b.to(DEVICE, dtype=torch.float32)
    y_np = y_b[0, 0].numpy()

    if y_np.max() > 0:
        dist = cv2.distanceTransform((1-(y_np>0.5).astype(np.uint8)), cv2.DIST_L2, 3)
        bnd_np, far_np = (dist>0)&(dist<=20), (dist>30)
    else:
        bnd_np = far_np = None

    for k, net in loaded.items():
        m_in = torch.zeros(1, 1, *SIZE, device=DEVICE, requires_grad=True)
        out  = net([x_b, m_in])

        feat = feat_store.get(k)
        if feat is not None and bnd_np is not None:
            h, w = feat.shape[2], feat.shape[3]
            bnd_s = cv2.resize(bnd_np.astype(np.uint8), (w,h), interpolation=cv2.INTER_NEAREST).astype(bool)
            far_s = cv2.resize(far_np.astype(np.uint8), (w,h), interpolation=cv2.INTER_NEAREST).astype(bool)
            fhw   = feat[0].permute(1,2,0).cpu().numpy()
            if bnd_s.sum()>0 and far_s.sum()>0:
                vb = fhw[bnd_s].mean(0); vf = fhw[far_s].mean(0)
                diag[k]["cos_sim"].append(float(np.dot(vb,vf)/(np.linalg.norm(vb)*np.linalg.norm(vf)+1e-8)))

        prob = torch.sigmoid(out).detach()
        diag[k]["saturation"].append(float(((prob<0.01)|(prob>0.99)).float().mean()))
        diag[k]["entropy"].append(float(-(prob*torch.log(prob+1e-8)+(1-prob)*torch.log(1-prob+1e-8)).mean()))
        out.sum().backward()
        diag[k]["grad_norm"].append(float(m_in.grad.norm(2)) if m_in.grad is not None else 0.0)
        net.zero_grad()

for k in loaded: hooks[k].remove()

# ── Print & Save Summary ──────────────────────────────────────────
print("\n=== DIAGNOSTIC SUMMARY ===")
headers = ["Model", "CosSim", "Saturation(%)", "Entropy", "GradNorm"]
print(f"{'Model':<20} {'CosSim':>8} {'Sat%':>10} {'Entropy':>9} {'GradNorm':>12}")
print("-"*65)
summary_out = {}
for k in loaded:
    d = diag[k]
    cos = np.mean(d["cos_sim"]) if d["cos_sim"] else float("nan")
    sat = np.mean(d["saturation"]) * 100
    ent = np.mean(d["entropy"])
    gn  = np.mean(d["grad_norm"])
    label = MODELS_INFO[k]["label"]
    summary_out[k] = {"label": label, "cos_sim": round(cos,4),
                      "saturation_pct": round(sat,2), "entropy": round(ent,4),
                      "grad_norm": round(gn,4)}
    print(f"{label:<20} {cos:>8.4f} {sat:>9.2f}% {ent:>9.4f} {gn:>12.4f}")

# Save JSON
with open(f"{DIAG_DIR}/diagnostic_phase7b.json", "w") as f:
    json.dump({"bn_drift": bn_summary, "model_metrics": summary_out}, f, indent=2)
print(f"\nSaved: {DIAG_DIR}/diagnostic_phase7b.json")

# ── Pass/Fail Decision ────────────────────────────────────────────
print("\n=== DECISION ===")
if "M12" in summary_out and "M11" in summary_out:
    sat_delta  = summary_out["M11"]["saturation_pct"] - summary_out["M12"]["saturation_pct"]
    gn_delta   = summary_out["M12"]["grad_norm"] - summary_out["M11"]["grad_norm"]
    kl_m11     = bn_summary.get("M11_vs_M00", {}).get("avg_kl", float("nan"))
    kl_m12     = bn_summary.get("M12_vs_M00", {}).get("avg_kl", float("nan"))

    print(f"  Saturation delta (M11-M12)  : {sat_delta:+.2f}pp  (want > 0)")
    print(f"  Grad Norm delta (M12-M11)   : {gn_delta:+.4f}    (want > 0)")
    print(f"  BN KL M11 vs M00            : {kl_m11:.4f}")
    print(f"  BN KL M12 vs M00            : {kl_m12:.4f}    (want << M11)")

    if sat_delta > 0 and gn_delta > 0 and kl_m12 < kl_m11:
        print("\n  [SUCCESS] Feedback Trap appears severed in M12!")
        print("  -> Proceed to Part 3 multi-seed to confirm with FP metrics.")
    else:
        print("\n  [INCONCLUSIVE] Some metrics don't improve as expected.")
        print("  -> Check loss curves and consider adjusting alpha/lr before multi-seed.")

---
## Part 3 — Multi-Seed Loop: M11 vs M12

> **INSTRUCTION:** Comment out the `raise` line below once Smoke Test (Part 1) and Diagnostics (Part 2) are confirmed OK. Then run this cell to kick off the full multi-seed comparison.

**Seeds:** `[42, 1337, 2024, 7, 99]`  
**Cells:** `M11` (Feedback Trap) vs `M12` (Fixed)  
**GPU time estimate:** ~2h per seed-pair on Kaggle T4 × 200ep

In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 3: MULTI-SEED LOOP — M11 (Trap) vs M12 (Fixed)
# COMMENT OUT THE GUARD BELOW AFTER SMOKE TEST PASSES
# ════════════════════════════════════════════════════════════════
raise RuntimeError(
    "GUARD: Remove this line after Part 1 & 2 succeed to run multi-seed."
)

# ─── Config ───────────────────────────────────────────────────────
SEEDS   = [42, 1337, 2024, 7, 99]
EPOCHS  = 200

CELL_CONFIGS = {
    "M11": {
        "model_kwargs": {"gate": "binary", "detach_feedback": False},
        "no_feedback": False,
        "loss_fn_cls": lambda: Phase6AsymmetricBCELoss(alpha=0.7, beta=0.3, lam=0.5),
        "label": "FB+Tversky[Trap]",
    },
    "M12": {
        "model_kwargs": {"gating_mode": "soft_or", "detach_feedback": True},
        "no_feedback": False,
        "loss_fn_cls": lambda: Phase6AsymmetricBCELoss(alpha=0.7, beta=0.3, lam=0.5),
        "label": "FB+Tversky[Fixed]",
    },
}

all_results = {}  # {"M12_seed42": {"bin_dice": ..., "bin_fpr": ...}, ...}

for cell_name, cell_cfg in CELL_CONFIGS.items():
    for seed in SEEDS:
        run_id   = f"{cell_name}_seed{seed}"
        ckpt_p   = f"{CKPT_DIR}/{run_id}.pth"
        log_p    = f"{LOG_DIR}/{run_id}.log"

        # Skip if already done (allow partial resume)
        if os.path.exists(ckpt_p):
            print(f"[SKIP] {run_id} checkpoint exists.")
            continue

        print(f"\n{'='*55}")
        print(f"Training {run_id} ({cell_cfg['label']})")
        print(f"{'='*55}")

        seeding(seed)
        (tr_x, tr_y), (vl_x, vl_y) = load_data(DATASET_PATH)
        tr_x, tr_y = shuffling(tr_x, tr_y)

        tr_loader = DataLoader(DATASET(tr_x, tr_y, SIZE, transform=aug),
                               batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        vl_loader = DataLoader(DATASET(vl_x, vl_y, SIZE, transform=None),
                               batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        net      = FANet(**cell_cfg["model_kwargs"]).to(DEVICE)
        loss_fn  = cell_cfg["loss_fn_cls"]()
        opt      = torch.optim.Adam(net.parameters(), lr=LR)
        sched    = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=5)
        tr_mask  = init_mask(tr_x, SIZE)
        vl_mask  = init_mask(vl_x, SIZE)
        best     = float('inf')
        no_fb    = cell_cfg["no_feedback"]

        for ep in range(EPOCHS):
            t0 = time.time()
            tl, tr_mask_new = train_epoch(net, tr_loader, tr_mask, opt, loss_fn, DEVICE, SIZE, no_fb)
            vl, vl_mask_new, bm = evaluate_epoch(net, vl_loader, vl_mask, loss_fn, DEVICE, SIZE, no_fb)
            sched.step(vl)
            if vl < best:
                best = vl
                torch.save(net.state_dict(), ckpt_p)
                tr_mask, vl_mask = tr_mask_new, vl_mask_new
            mins, secs = epoch_time(t0, time.time())
            row = (f"Ep {ep+1:03}/{EPOCHS} | {mins}m{secs}s | "
                   f"TrLoss={tl:.4f} VlLoss={vl:.4f} | "
                   f"Dice={bm['bin_dice']:.4f} FPR={bm['bin_fpr']:.4f}")
            print_and_save(log_p, row)

        all_results[run_id] = {"cell": cell_name, "seed": seed,
                               "best_vl_loss": round(best, 4), **bm}
        print(f"[DONE] {run_id} | best_loss={best:.4f}")

# ── Save aggregate results ────────────────────────────────────────
with open(f"{RESULT_DIR}/multiseed_summary.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nAll runs complete. Results: {RESULT_DIR}/multiseed_summary.json")

# ── Quick summary table ───────────────────────────────────────────
print(f"\n{'Run ID':<25} {'Dice':>8} {'FPR':>8} {'Prec':>8} {'Rec':>8}")
print("-" * 62)
for rid, r in all_results.items():
    print(f"{rid:<25} {r['bin_dice']:>8.4f} {r['bin_fpr']:>8.4f} "
          f"{r['bin_prec']:>8.4f} {r['bin_rec']:>8.4f}")